In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np
import pickle
import matplotlib.pyplot as plt
import os

## Caricamento dei files

In [ ]:
# Imposta il percorso della cartella contenente i file pkl
path = '/content/drive/MyDrive/cold_heads_monitoring/Dataset/'
arrays = []

# Itera su tutti i file pkl nella cartella e caricali in una lista
for filename in os.listdir(path):
    print(filename)
    if filename.endswith('.pkl'):
        with open(os.path.join(path, filename), 'rb') as f:
            # Carica l'array NumPy dal file e seleziona solo il secondo canale
            arr = np.asarray(pickle.load(f))[:, 2:3, :]
            arrays.append(arr)


## Bilanciamento del dataset

In [ ]:
import sys
# Calcola la dimensione minima tra gli array nella lista
#min_len = min([arr.shape[0] for arr in arrays])
min_len = 2500

# Bilancia gli array nella lista
balanced_arrays = [arr[:min_len] for arr in arrays]

head_new = np.squeeze(np.asarray(balanced_arrays[0]),1)
dataset_no_comp = np.squeeze(np.asarray(balanced_arrays[1]),1)
#dataset_no_hel = np.squeeze(np.asarray(balanced_arrays[2]),1)
head_30k_ore = np.squeeze(np.asarray(balanced_arrays[2]),1)

head_new.shape, dataset_no_comp.shape, head_30k_ore.shape

In [ ]:
data = np.concatenate([head_new,dataset_no_comp,head_30k_ore], axis=0)
np.random.shuffle(data)
data.shape

## Training DBSCAN

In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN

class IncrementalDBSCAN:
    def __init__(self, epsilon, min_samples):
        self.epsilon = epsilon
        self.min_samples = min_samples
        self.dbscan = None

    def fit(self, X):
        self.dbscan = DBSCAN(eps=self.epsilon, min_samples=self.min_samples)
        self.dbscan.fit(X)
        dim  = len(self.dbscan.components_)
        plt.scatter(self.dbscan.components_[:, 0], self.dbscan.components_[:, 1], c=self.dbscan.labels_[0:dim], cmap='viridis')
        plt.show()

    def incremental_dbscan(self,dbscan_model, new_data, eps, min_samples):
      # Estrai i componenti dal modello DBSCAN esistente
      components = dbscan_model.components_
      core_samples = dbscan_model.core_sample_indices_

      # Aggiungi i nuovi dati ai componenti esistenti
      updated_components = np.concatenate((components, new_data))
      updated_core_samples = np.concatenate((core_samples, np.arange(len(components), len(components) + len(new_data))))

      # Crea un nuovo modello DBSCAN con i componenti aggiornati
      updated_dbscan = DBSCAN(eps=eps, min_samples=min_samples)
      updated_dbscan.components_ = updated_components
      updated_dbscan.core_sample_indices_ = updated_core_samples

      # Esegui il clustering sui dati aggiornati
      updated_labels = updated_dbscan.fit_predict(updated_components)

      plt.scatter(updated_dbscan.components_[:, 0], updated_dbscan.components_[:, 1], c=updated_labels[:], cmap='viridis')
      plt.show()

      # Restituisci le etichette di clustering aggiornate
      return updated_labels

In [ ]:
dbscan = IncrementalDBSCAN(1, 10)
dbscan.fit(np.concatenate([head_30k_ore,dataset_no_comp], axis=0))

# Ottenere le etichette di clustering iniziali
initial_labels = dbscan.dbscan.labels_
n_clusters = len(set(initial_labels)) - (1 if -1 in initial_labels else 0)
print("Numero di cluster trovati:", n_clusters)

n_anomalies = list(initial_labels).count(-1)
print("Numero di punti anomali trovati:", n_anomalies)



In [ ]:
# Aggiornamento con nuovi dati
new_data = head_new[0:1500]
updated_labels = dbscan.incremental_dbscan(dbscan.dbscan,new_data,1,5)

# Ottenere le etichette di clustering aggiornate
n_clusters_2 = len(set(updated_labels)) - (1 if -1 in updated_labels else 0)
print("Numero di cluster trovati nell'aggiornamento successivo:", n_clusters_2)


n_anomalies_2 = list(updated_labels).count(-1)
print("Numero di punti anomali trovati nell'aggiornamento successivo:", n_anomalies_2)
